# Component 3 - XAI step 4: expert oversight

The expert corrects a pipeline result **without re-running any model**: exclude cells, remove or add ROI area.
The numbers are recomputed from the saved segmentation, every review is logged (append-only), and the original
pipeline result is never changed.

* **Review queue**: RED cells first, then YELLOW (least confident first) - where step 2 found the errors.
* **Evaluation** (CPU, from the step 2 crop results): a simulated reviewer who excludes all RED + YELLOW cells -
  does the error vs the manual values go down? (paired Wilcoxon, Holm; rules in `config.yaml -> xai.oversight`).

**GPU runtime** for cells 4-6 (the pipeline); cell 3 is CPU.


In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" checkout -- ml/cellular/notebooks/
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Install StarDist + unit tests
!pip -q install tensorflow tf-keras stardist
!python -m pytest tests/test_oversight.py tests/test_pipeline.py -q


In [ ]:
# 3. Simulated reviewer on the 40 crops (CPU, seconds; needs step 2's --crops results)
!python -m oversight --simulate


In [ ]:
# 4. The pipeline only saves more files now - its numbers must still equal Phase 6
!python -m pipeline --check 10 2>&1 | grep -v -E "Warning|warn|oneDNN"


In [ ]:
# 5. Analyse one image (bubbles in the field) and show its review queue
import pandas as pd
from IPython.display import Image, display
from config import load_config
IMAGES_DIR = load_config()["paths"]["images_dir"]
man = pd.read_csv("splits/manifest.csv").set_index("image_id")
IID = "NEB-2021-03-13084 R_SP"
IMG = f"{IMAGES_DIR}/{man.loc[IID, 'image_file']}"
!python -m pipeline "{IMG}" 2>&1 | grep -v -E "Warning|warn|oneDNN"
OUT = f"{OUTPUT_ROOT}/pipeline"
display(Image(f"{OUT}/{IID}_review_queue.png", width=1200))
pd.read_csv(f"{OUT}/{IID}_review_queue.csv").head(15)


In [ ]:
# 6. An example review: remove the central bubble area, exclude cell 761 (on the right-hand bubble)
import json
edits = {"reviewer": "demo",
         "edits": [{"action": "remove_area", "polygon": [[600, 330], [720, 330], [720, 450], [600, 450]],
                    "reason": "bubble"},
                   {"action": "exclude_cells", "cell_ids": [761], "reason": "on a bubble"}]}
open(f"{OUT}/{IID}_edits_demo.json", "w").write(json.dumps(edits, indent=2))
!python -m oversight "{OUT}/{IID}.json" --edits "{OUT}/{IID}_edits_demo.json" --image "{IMG}"
display(Image(f"{OUT}/{IID}_reviewed_overlay.png", width=1200))
print(open(f"{OUT}/{IID}_edits.jsonl").read())
